# Pull player positions (for position-relative box-score comparison)

**Before running:** same kernel as the other pipeline notebooks (needs pandas, numpy, nba_api).

Basketball-Reference's Player Totals table has a `Pos` column, but it's a 500+ row table per season and scraping it came back truncated (~128 of 500+ rows) -- not reliable. This instead uses nba_api's `commonteamroster` endpoint, which returns each team's roster (with position) directly from the NBA's own data, no scraping involved.

This makes **one call per team per season** (30 teams x 9 seasons = 270 calls) rather than one bulk call, so it takes noticeably longer than the earlier pulls -- expect roughly 8-12 minutes with polite rate-limiting between calls.

Output: `player_positions.csv` in this folder, with one row per (PLAYER_ID, Season) giving a bucketed G/F/C position. A traded player may show up via two different teams' rosters that season; position is stable regardless of team, so duplicates are just deduplicated.

In [ ]:
import pandas as pd
import time
import os
from nba_api.stats.endpoints import commonteamroster
from nba_api.stats.static import teams as static_teams

REQUEST_TIMEOUT = 30
MAX_RETRIES = 4
RETRY_BACKOFF_BASE = 2.0
CALL_DELAY = 0.7  # be polite between calls across 270 of them

SEASONS = ["2017-18","2018-19","2019-20","2020-21","2021-22","2022-23","2023-24","2024-25","2025-26"]

def season_label_year(season):
    return int(season.split("-")[0])

all_teams = static_teams.get_teams()
team_ids = [t["id"] for t in all_teams]
print(f"{len(team_ids)} teams x {len(SEASONS)} seasons = {len(team_ids)*len(SEASONS)} calls")

def pull_roster(team_id, season):
    last_exc = None
    for attempt in range(MAX_RETRIES):
        try:
            resp = commonteamroster.CommonTeamRoster(team_id=team_id, season=season, timeout=REQUEST_TIMEOUT)
            df = resp.get_data_frames()[0]
            return df
        except Exception as e:
            last_exc = e
            wait = RETRY_BACKOFF_BASE * (2 ** attempt)
            print(f"  team {team_id} {season} attempt {attempt+1} failed: {e!r} -- retrying in {wait:.1f}s")
            time.sleep(wait)
    raise last_exc

In [ ]:
rows = []
for season in SEASONS:
    label = season_label_year(season)
    print(f"Season {season}...")
    for team_id in team_ids:
        df = pull_roster(team_id, season)
        if not df.empty:
            df = df[["PLAYER_ID","PLAYER","POSITION"]].copy()
            df["Season"] = label
            rows.append(df)
        time.sleep(CALL_DELAY)
    print(f"  done, {sum(len(r) for r in rows)} rows so far")

positions_raw = pd.concat(rows, ignore_index=True)
print(f"\nTotal roster rows: {len(positions_raw)}")
positions_raw.head()

In [ ]:
def bucket_position(pos):
    if not isinstance(pos, str) or not pos:
        return None
    first = pos.split("-")[0].strip().upper()
    if first.startswith("G"):
        return "G"
    if first.startswith("F"):
        return "F"
    if first.startswith("C"):
        return "C"
    return None

positions_raw["PosBucket"] = positions_raw["POSITION"].apply(bucket_position)
print("Unmapped position strings:", positions_raw[positions_raw["PosBucket"].isna()]["POSITION"].unique())

# collapse to one row per (PLAYER_ID, Season) -- take the most common bucket if a
# traded player's two team rosters disagree (rare, but be defensive)
positions = (
    positions_raw.dropna(subset=["PosBucket"])
    .groupby(["PLAYER_ID","Season"])["PosBucket"]
    .agg(lambda s: s.mode().iloc[0])
    .reset_index()
)
print(f"Unique (PLAYER_ID, Season) position rows: {len(positions)}")
print(positions["PosBucket"].value_counts())

out_path = os.path.join(os.getcwd(), "player_positions.csv")
positions.to_csv(out_path, index=False)
print(f"Wrote {out_path}")